# Physical Bullying Detection Challenge (RWF-2000 + VideoMAE)

This notebook has two parts:

1. **MENTOR SETUP** (top section, clearly marked) — downloads the dataset,
   splits it into a labeled training set and an **unlabeled** validation
   set, and saves the true validation labels separately for grading.
   **Delete this whole section before handing the notebook to participants.**
2. **PARTICIPANT SECTION** — what participants actually work with. It
   loads the labeled training data and the unlabeled validation data, gives
   a deliberately weak sample model as a working starting point, and ends
   by writing `submission.csv` with predictions for the validation set.

A small **MENTOR GRADING** cell is included at the very bottom, to be run
by the mentor (not the participant) after a `submission.csv` has been
produced, to score it against the hidden ground truth.

Pipeline: Video → extract 16 frames → resize 224x224 → RGB →
VideoMAEImageProcessor → VideoMAE model → Fight / NonFight

A GPU runtime is recommended (Runtime → Change runtime type → GPU) but
the notebook also runs on CPU, just slower.

## Section 1 - Install Libraries

In [2]:
!pip install torch torchvision
!pip install transformers
!pip install opencv-python-headless
!pip install scikit-learn
!pip install matplotlib
!pip install pandas
!pip install tqdm
!pip install kagglehub



[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


## Section 2 - Import Libraries

In [1]:
# Standard library
import os
import csv
import glob
import shutil
import random

# Numerical / data handling
import numpy as np
import pandas as pd

# Video and image handling
import cv2

# Plotting
import matplotlib.pyplot as plt

# PyTorch
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

# Hugging Face Transformers (VideoMAE model + processor)
from transformers import VideoMAEImageProcessor, VideoMAEForVideoClassification

# Metrics
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix

# Progress bars
from tqdm.auto import tqdm

print("All libraries imported successfully.")


2026-07-12 14:00:48.121736: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-12 14:00:48.129681: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1783854048.139615    8451 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1783854048.142654    8451 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1783854048.150425    8451 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking 

All libraries imported successfully.


## Section 3 - Configuration

Shared settings used by both the mentor setup and the participant section.

In [2]:
# ----------------------------
# General settings
# ----------------------------
SEED = 42               # random seed, for reproducibility
IMAGE_SIZE = 224          # frames are resized to IMAGE_SIZE x IMAGE_SIZE
NUM_FRAMES = 16            # number of frames sampled from each video
BATCH_SIZE = 4             # videos per training batch
LEARNING_RATE = 2e-5          # learning rate for the optimizer

# ----------------------------
# Label mapping
# ----------------------------
LABEL2ID = {"NonFight": 0, "Fight": 1}
ID2LABEL = {v: k for k, v in LABEL2ID.items()}

# ----------------------------
# Reproducibility
# ----------------------------
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# ----------------------------
# GPU check
# ----------------------------
if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
    print("GPU detected. Using device:", torch.cuda.get_device_name(0))
else:
    DEVICE = torch.device("cpu")
    print("No GPU detected. Using device: CPU (training will be slower).")


GPU detected. Using device: NVIDIA GeForce RTX 3060


### Download the dataset

Tries an automatic Kaggle mirror first. If that fails, set `DATA_DIR` by
hand to a folder that already contains `train/Fight`, `train/NonFight`,
`val/Fight`, `val/NonFight` subfolders, then skip straight to the next
cell.

In [14]:
DATA_DIR = None

def _find_archive(cache_root, min_size_bytes=500 * 1024 * 1024):
    '''Look for an already-downloaded archive in the kagglehub cache.'''
    candidates = []
    for root, dirs, files in os.walk(cache_root):
        for fname in files:
            fpath = os.path.join(root, fname)
            try:
                size = os.path.getsize(fpath)
            except OSError:
                continue
            if size >= min_size_bytes and zipfile.is_zipfile(fpath):
                candidates.append(fpath)
    return candidates

def _fix_filename(raw_name):
    '''Repair a file name that was saved with a non-UTF-8 codepage.'''
    try:
        raw_bytes = raw_name.encode("cp437")
    except Exception:
        return None
    for encoding in ("gbk", "big5", "utf-8"):
        try:
            return raw_bytes.decode(encoding)
        except Exception:
            continue
    return None

def _safe_extract(zip_path, target_dir):
    '''Extract a zip, giving any unsafe file name a simple replacement
    instead of skipping it, so no data is lost.'''
    os.makedirs(target_dir, exist_ok=True)
    renamed = 0
    with zipfile.ZipFile(zip_path) as zf:
        for counter, info in enumerate(tqdm(zf.infolist(), desc="Extracting")):
            parts = info.filename.split("/")
            fname = parts[-1]
            if fname == "":
                continue  # directory entry, nothing to write
            safe_name = fname
            if len(fname.encode("utf-8", errors="replace")) > 200:
                fixed = _fix_filename(fname)
                if fixed and len(fixed.encode("utf-8")) <= 200:
                    safe_name = fixed
                else:
                    ext = os.path.splitext(fname)[1] or ".avi"
                    safe_name = f"video_{counter}{ext}"
                renamed += 1
            out_dir = os.path.join(target_dir, *parts[:-1])
            os.makedirs(out_dir, exist_ok=True)
            try:
                with zf.open(info) as src, open(os.path.join(out_dir, safe_name), "wb") as dst:
                    dst.write(src.read())
            except Exception as e:
                print("Skipping an entry that could not be extracted:", e)
    print("Extraction finished. Renamed", renamed, "broken file name(s).")

import zipfile

try:
    import kagglehub
    downloaded_path = kagglehub.dataset_download("vulamnguyen/rwf2000")
    print("Downloaded dataset to:", downloaded_path)
    DATA_DIR = downloaded_path
except Exception as error:
    print("Automatic download failed:", error)
    print("Trying to repair a partially-cached copy instead...")
    cache_root = os.path.expanduser("~/.cache/kagglehub/datasets/vulamnguyen/rwf2000")
    archives = _find_archive(cache_root)
    if archives:
        REPAIR_TARGET_DIR = "./RWF-2000-data"
        _safe_extract(archives[0], REPAIR_TARGET_DIR)
        DATA_DIR = REPAIR_TARGET_DIR
    else:
        print("Could not download or repair automatically.")
        print("Set DATA_DIR by hand to a folder containing train/ and val/ subfolders.")

def _find_dataset_root(start_path):
    '''Some mirrors nest the data one level deeper. Find the folder that
    directly contains train/ and val/ subfolders.'''
    for root, dirs, files in os.walk(start_path):
        if "train" in dirs and "val" in dirs:
            return root
    return start_path

DATASET_ROOT = _find_dataset_root(DATA_DIR)
print("Dataset root:", DATASET_ROOT)
print("Contents:", os.listdir(DATASET_ROOT))


Download already complete (12317041801 bytes).
Extracting files...
Automatic download failed: [Errno 36] File name too long: '/home/s4lm-xi/.cache/kagglehub/datasets/vulamnguyen/rwf2000/versions/1/RWF-2000/train/Fight/1060314╤äΓòò╨╜╤à╨│╨▓╤à╨┤╨╣╤å╨⌐╨»╤ë╨û╨╗╤ë╨⌐╨▓╤å╨É╨╡╤ê╨╕Γòæ╤à╨╛╨┤╤å╨¬Γöñ╤à╨Ü╨½╤äΓòæ╨¢╤äΓòùΓòó╤ÅΓò¥╨ÿ╤à╨¥╨«╤ë╨ù╨í╤àΓöéΓûæ╤ÅΓò¥╨Ö_17.avi'
Trying to repair a partially-cached copy instead...


Extracting:   0%|          | 0/2000 [00:00<?, ?it/s]

Extraction finished. Renamed 26 broken file name(s).
Dataset root: ./RWF-2000-data/RWF-2000
Contents: ['train', 'val']


### Build the participant-facing train/val split and hide the val labels

Reuses RWF-2000's existing `train/` and `val/` folders as-is. The train
split stays labeled (participants need labels to train). The val split is
copied into a flat, anonymized, shuffled folder with the labels stripped
out, and the true labels are saved to a mentor-only CSV for grading.

In [16]:
DATASET_ROOT

'./RWF-2000-data/RWF-2000'

In [17]:
def list_videos(split_name):
    '''(video_path, label) pairs for a given split, read from DATASET_ROOT.'''
    items = []
    for class_name, label in LABEL2ID.items():
        class_dir = os.path.join(DATASET_ROOT, split_name, class_name)
        paths = glob.glob(os.path.join(class_dir, "*.avi")) + \
                glob.glob(os.path.join(class_dir, "*.mp4"))
        for p in paths:
            items.append((p, label))
    return items

_train_items = list_videos("train")
_val_items = list_videos("val")
print("Train videos found:", len(_train_items))
print("Val videos found:", len(_val_items))

# Participants train directly on the original labeled train/ folder.
PARTICIPANT_TRAIN_DIR = os.path.join(DATASET_ROOT, "train")

# Participants get an anonymized, unlabeled copy of val/.
PARTICIPANT_VAL_DIR = ",/participant_val"
MENTOR_ONLY_DIR = ".//mentor_only"
MENTOR_GROUND_TRUTH_CSV = os.path.join(MENTOR_ONLY_DIR, "ground_truth.csv")

os.makedirs(PARTICIPANT_VAL_DIR, exist_ok=True)
os.makedirs(MENTOR_ONLY_DIR, exist_ok=True)

random.shuffle(_val_items)  # shuffle so file order can't leak the label

ground_truth_rows = []
for i, (src_path, label) in enumerate(tqdm(_val_items, desc="Anonymizing val set")):
    ext = os.path.splitext(src_path)[1]
    video_id = f"val_{i:04d}{ext}"
    dst_path = os.path.join(PARTICIPANT_VAL_DIR, video_id)
    shutil.copyfile(src_path, dst_path)
    ground_truth_rows.append((video_id, ID2LABEL[label]))

with open(MENTOR_GROUND_TRUTH_CSV, "w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow(["video_id", "true_label"])
    writer.writerows(ground_truth_rows)

print()
print("Participant train dir (labeled):  ", PARTICIPANT_TRAIN_DIR)
print("Participant val dir (unlabeled):  ", PARTICIPANT_VAL_DIR, "-", len(ground_truth_rows), "videos")
print("Ground truth saved (mentor only): ", MENTOR_GROUND_TRUTH_CSV)


Train videos found: 1600
Val videos found: 400


Anonymizing val set:   0%|          | 0/400 [00:00<?, ?it/s]


Participant train dir (labeled):   ./RWF-2000-data/RWF-2000/train
Participant val dir (unlabeled):   ,/participant_val - 400 videos
Ground truth saved (mentor only):  .//mentor_only/ground_truth.csv
